In [1]:
import re
import os
import csv
import json
import pandas as pd

# klines 디렉토리 내의 파일 탐색
timeframe = "15m"
klines_file_path = f"../data/binance/futures/um/monthly/klines/BTCUSDT/{timeframe}"
klines_use_columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_volume",
    "count",
    "taker_buy_volume",
    "taker_buy_quote_volume",
]
klines_file_list = os.listdir(klines_file_path)
klines_file_list.sort()

# position distribution(pd) 디렉토리 내의 파일 탐색
pd_file_path = "../data/binance/futures/um/monthly/position_distribution/900000"
pd_file_list = os.listdir(pd_file_path)
pd_file_list.sort()

# 저장할 데이터 프레임 생성
result_path = "../data/binance/futures/um/dataset/"
retail_result_df = None
institutional_result_df = None
action_dims = 50

# l_i, s_i column 순서 정렬
column_order = []
for i in range(action_dims):
    column_order.extend([f"l_{i}"])
for i in range(action_dims):
    column_order.extend([f"s_{i}"])

for pd_file in pd_file_list:
    # pd_df 읽기
    pd_df = pd.read_csv(os.path.join(pd_file_path, pd_file), index_col=0)

    # JSON 형태로 저장된 리스트 컬럼 변환
    pd_df["retail_long"] = pd_df["retail_long"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["retail_short"] = pd_df["retail_short"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["institutional_long"] = pd_df["institutional_long"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["institutional_short"] = pd_df["institutional_short"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )

    # 날짜/시간 열을 인덱스로 설정
    pd_df.index = pd.to_datetime(pd_df.index)

    # retail_pd_df와 institutional_pd_df 생성
    retail_pd_df = pd.DataFrame(index=pd_df.index)
    institutional_pd_df = pd.DataFrame(index=pd_df.index)

    # 리스트 데이터를 개별 컬럼으로 확장
    for i in range(action_dims):
        retail_pd_df[f"l_{i}"] = pd_df["retail_long"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        retail_pd_df[f"s_{i}"] = pd_df["retail_short"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        institutional_pd_df[f"l_{i}"] = pd_df["institutional_long"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        institutional_pd_df[f"s_{i}"] = pd_df["institutional_short"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )

    retail_pd_df = retail_pd_df[column_order]
    institutional_pd_df = institutional_pd_df[column_order]

    # pd_df file 이름에서 년도, 월 추출
    match = re.search(r"BTCUSDT-pd-(\d{4})-(\d{2})\.csv", pd_file)
    if not match:
        print(f"파일명 형식이 맞지 않습니다: {pd_file}")
        continue
    year, month = match.groups()

    # 년도, 월에 따라서 klines_file_list에서 해당 파일 찾기
    klines_file = [
        file for file in klines_file_list if file.endswith(f"{year}-{month}.csv")
    ][0]
    klines_df = pd.read_csv(
        os.path.join(klines_file_path, klines_file),
        index_col=0,
        usecols=klines_use_columns,
    )

    # open_time을 datetime으로 변환, 30분 뒤로 밀기
    klines_df.index = pd.to_datetime(klines_df.index, unit="ms")
    klines_df.index = klines_df.index + pd.Timedelta(minutes=int(timeframe[:-1]))

    # klines_df와 retail_pd_df, institutional_pd_df 결합
    retail_combined_df = pd.concat([klines_df, retail_pd_df], axis=1)
    institutional_combined_df = pd.concat([klines_df, institutional_pd_df], axis=1)

    if retail_result_df is None:
        retail_result_df = retail_combined_df
    else:
        retail_result_df = pd.concat([retail_result_df, retail_combined_df])

    if institutional_result_df is None:
        institutional_result_df = institutional_combined_df
    else:
        institutional_result_df = pd.concat(
            [institutional_result_df, institutional_combined_df]
        )

# 9:1 비율로 train/test 데이터 분리
ratio = 0.9
retail_train_df = retail_result_df.iloc[: int(len(retail_result_df) * ratio)]
retail_test_df = retail_result_df.iloc[int(len(retail_result_df) * ratio) :]

institutional_train_df = institutional_result_df.iloc[
    : int(len(institutional_result_df) * ratio)
]
institutional_test_df = institutional_result_df.iloc[
    int(len(institutional_result_df) * ratio) :
]

# 디렉토리 생성
os.makedirs(result_path, exist_ok=True)

# 각각의 데이터프레임을 CSV 파일로 저장
retail_train_df.to_csv(
    os.path.join(result_path, f"retail_train_{timeframe}.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
retail_test_df.to_csv(
    os.path.join(result_path, f"retail_test_{timeframe}.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
institutional_train_df.to_csv(
    os.path.join(result_path, f"institutional_train_{timeframe}.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
institutional_test_df.to_csv(
    os.path.join(result_path, f"institutional_test_{timeframe}.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)

In [2]:
retail_train_df

,open,high,low,close,volume,quote_volume,count,taker_buy_volume,taker_buy_quote_volume,l_0,...,s_40,s_41,s_42,s_43,s_44,s_45,s_46,s_47,s_48,s_49
2022-06-01 00:15:00,31797.9,31880.0,31704.0,31755.1,3115.705,9.902437e+07,29916,1328.900,4.225072e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022-06-01 00:30:00,31755.0,31868.6,31680.0,31812.8,4443.544,1.412482e+08,39724,2418.946,7.690107e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022-06-01 00:45:00,31812.8,31986.1,31781.8,31936.5,3914.311,1.248437e+08,37277,2187.604,6.979138e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022-06-01 01:00:00,31936.6,31970.0,31876.5,31925.5,2115.328,6.752406e+07,25545,959.438,3.062966e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022-06-01 01:15:00,31925.5,31938.6,31868.0,31904.4,1443.033,4.601879e+07,17946,658.890,2.101270e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-18 11:00:00,104475.0,104719.4,104400.0,104499.1,2373.586,2.481713e+08,36859,1549.337,1.620088e+08,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-18 11:15:00,104499.1,104697.5,104455.1,104624.1,1186.949,1.241880e+08,25102,585.045,6.120962e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-18 11:30:00,104624.2,104763.2,104550.2,104698.3,1105.728,1.157192e+08,23419,547.412,5.729823e+07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-18 11:45:00,104698.2,104957.4,104643.0,104843.1,1892.149,1.983781e+08,32780,1066.847,1.118473e+08,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
